# Deepfake Detection — Colab Training Runner

Thin runner notebook: clones the project repo, installs extra deps, authenticates Kaggle, then runs the real pipeline scripts that live in `src/` (download → subset → train → evaluate → Grad-CAM). All the actual logic is in the repo's `.py` files, not in this notebook — this just drives them on Colab's free GPU.

**Before running:** Runtime → Change runtime type → GPU (T4).

**Kaggle auth (one-time setup):**
1. Go to https://www.kaggle.com/settings → API → "Create New Token" (downloads `kaggle.json`).
2. Open `kaggle.json`, note the `username` and `key` values.
3. In this Colab notebook, click the key icon (🔑) in the left sidebar → add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` → toggle "Notebook access" on for both.

You only need to do this once per Google account — the secrets persist across sessions.

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
REPO_URL = "https://github.com/RohanTrivedi09/deepfakecomputervision.git"
REPO_DIR = "deepfakecomputervision"

import os
if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone $REPO_URL
    %cd {REPO_DIR}

!pwd

## 2. Install extra dependencies

Colab already ships CUDA-enabled `torch`/`torchvision` — reinstalling them from `requirements.txt` would risk swapping in a CPU build, so we only install what's missing: `kaggle` (dataset download) and `grad-cam` (explainability).

In [ ]:
!pip install -q kaggle grad-cam

## 3. Kaggle authentication (from Colab Secrets)

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Kaggle credentials loaded for user:", os.environ["KAGGLE_USERNAME"])

## 4. Download the dataset

Downloads the full 140k Real and Fake Faces dataset into `data/raw/` (skips automatically if already present — safe to re-run).

In [ ]:
!python -m src.preprocessing.download_data

## 5. Build the stratified subset

Defaults to 10k/1k/1k real+fake images per train/valid/test split (12k per class, 24k total) — within the plan's 10–15k/class v1 range. Adjust the flags below if you want a larger or smaller subset.

In [ ]:
!python -m src.preprocessing.build_subset \
    --per-class-train 10000 \
    --per-class-valid 1000 \
    --per-class-test 1000

## 6. Train

Fine-tunes ResNet18 (ImageNet-pretrained). Only the best-val-accuracy checkpoint is kept, saved to `models/checkpoints/best_model.pth`. ~8 epochs on a T4 with a 24k-image subset should take well under an hour.

In [ ]:
!python -m src.training.train --epochs 8 --batch-size 32 --lr 1e-4

## 7. Evaluate on the held-out test split

In [ ]:
!python -m src.evaluation.evaluate

## 8. Generate Grad-CAM overlays (correct + incorrect samples)

In [ ]:
!python -m src.evaluation.gradcam

## 9. Bring the results back to your local machine

Everything that matters is now sitting in the cloned repo on the Colab VM: `models/checkpoints/best_model.pth`, `reports/metrics.json`, `reports/figures/confusion_matrix.png`, `reports/figures/gradcam/*.png`. Two ways to get them back:

**Option A — commit + push from Colab (recommended, one command):**
Requires a GitHub Personal Access Token stored as a Colab secret (`GITHUB_TOKEN`, scope: repo). Then on your laptop you just `git pull`.

**Option B — manual download:**
Run the cell below to zip the results and download them via the browser, then unzip into the same paths in your local clone.

In [ ]:
# --- Option A: commit + push results back to GitHub ---
# from google.colab import userdata
# token = userdata.get("GITHUB_TOKEN")
# push_url = REPO_URL.replace("https://", f"https://{token}@")
# !git add models/checkpoints/best_model.pth reports/
# !git commit -m "Add trained checkpoint, metrics, and Grad-CAM samples from Colab"
# !git push $push_url HEAD:main

In [ ]:
# --- Option B: zip + browser download ---
import shutil
shutil.make_archive("/content/colab_results", "zip", ".", "models/checkpoints")
shutil.make_archive("/content/colab_reports", "zip", ".", "reports")
from google.colab import files
files.download("/content/colab_results.zip")
files.download("/content/colab_reports.zip")

## 10. Locally

Once `models/checkpoints/best_model.pth` exists in your local clone (via `git pull` after Option A, or unzipping the Option B downloads into place), run the Streamlit demo:

```bash
uv venv && source .venv/bin/activate
uv pip install -r requirements.txt
streamlit run app/streamlit_app.py
```